# miRNA-seq Processing and Analysis

**Tier 3 — Applied Bioinformatics | Module 35 · Notebook 1**

*Prerequisites: Module 01 (NGS Fundamentals), Module 03 (RNA-seq Analysis)*

---

**By the end of this notebook you will be able to:**
1. Describe miRNA biogenesis: Drosha/DGCR8 → pre-miRNA → Dicer → mature miRNA
2. Process small RNA-seq: adapter trimming, size selection, genome alignment
3. Quantify miRNA expression using miRBase annotation and featureCounts
4. Perform differential miRNA expression analysis with DESeq2
5. Predict miRNA target genes and perform pathway enrichment analysis


**Key resources:**
- [miRBase database](https://www.mirbase.org/)
- [miRDeep2 documentation](https://github.com/rajewsky-lab/mirdeep2)
- [TargetScan](https://www.targetscan.org/)
- [mirTarBase](https://mirtarbase.cuhk.edu.cn/)

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Module Overview](../README.md) | [Next: lncRNA & ncRNA Classification →](./02_lncrna_classification.ipynb)

---

## 1. miRNA Biology and Biogenesis

> Primary miRNA (pri-miRNA) → pre-miRNA hairpin → mature ~22 nt miRNA. RISC complex and mRNA silencing via 3'UTR seed region (positions 2-7). miRNA families. Star strand (*) vs guide strand.

## 2. Small RNA-seq Library Design

> 3' adapter ligation chemistry. Size selection (15-30 nt). Why standard RNA-seq fails for miRNA. Unique molecular identifiers (UMI) for deduplication. Minimum read depth (5–10M per sample).

## 3. Processing Pipeline

> Cutadapt for 3' adapter trimming. Length filtering (16-28 nt). Alignment with Bowtie (allow 0-1 mismatch). miRBase GFF annotation. Count with featureCounts.

In [ ]:
# Example: Adapter trimming and alignment
# !cutadapt -a TGGAATTCTCGGGTGCCAAGG -m 16 -M 28 -o trimmed.fastq.gz sample.fastq.gz
# !bowtie -x hg38 -p 8 --norc trimmed.fastq.gz -S aligned.sam
# !featureCounts -a miRBase_hg38.gff -o counts.txt aligned.bam

## 4. Differential miRNA Expression

> DESeq2 for count-based DE testing (same as RNA-seq). MA plot. Top differentially expressed miRNAs. Heatmap of DE miRNAs. miRNA naming convention (miR-21 vs miR-21-5p).

## 5. Target Prediction and Network Analysis

> TargetScan context++ scores. miRTarBase curated experimental targets. miRNA-target interaction network in NetworkX/Cytoscape. KEGG pathway enrichment of targets. miRNA sponge hypothesis and ceRNA networks.